## Sample-adequacy analysis

In [ ]:
"""
Sample-adequacy analysis for the final model (CB-SEM; MacCallum, Browne & Sugawara, 1996)
Finalc:\Users\1886199\OneDrive - UET\Drive D\Post-doc material\Conceptual frameworks\WP5\Paper_5\Alternate model\Data analysis\Alternate5_2.xlsx model: 11 constructs, 48 indicators, N = 405.
Computes model df, free parameters, RMSEA-based power and minimum N, plus
regression-based minimum N for each endogenous equation. Requires: scipy
"""
from scipy.stats import ncx2, ncf, f as f_dist
from scipy.optimize import brentq
from math import ceil

# True  -> noncentrality = (N - 1) * df * eps^2  (MacCallum et al., 1996)
# False -> noncentrality = N * df * eps^2        (convention of the earlier manuscript draft)
USE_N_MINUS_1 = True
# 'cohen'  -> lambda = f2 * (u + v + 1)  (Cohen, 1988; gives 102 for 3 tested of 6 predictors)
# 'gpower' -> lambda = f2 * N            (G*Power default)
REG_LAMBDA = "cohen"

# ---------------- 1. CORE FUNCTIONS ----------------
def sem_df(items, exog, endo, paths):
    """
    items: {construct: number_of_indicators}; exog/endo: exogenous / endogenous latents;
    paths: list of (source, target). Assumes marker-variable scaling, no cross-loadings,
    no correlated errors, uncorrelated disturbances, all exogenous latents correlated.
    """
    for s, t in paths:
        assert s in items and t in items, f"Unknown construct in path {s}->{t}"
        assert t in endo, f"Path target {t} must be endogenous"
    k, p = len(items), sum(items.values())
    moments = p * (p + 1) // 2
    measurement = (p - k) + p                      # free loadings + error variances
    ne = len(exog)
    latent = ne * (ne + 1) // 2 + len(endo) + len(paths)
    npar = measurement + latent
    cfa_npar = measurement + k * (k + 1) // 2
    return {"indicators": p, "constructs": k, "moments": moments, "npar": npar,
            "df": moments - npar, "cfa_npar": cfa_npar, "cfa_df": moments - cfa_npar}

def _m(n):
    return (n - 1) if USE_N_MINUS_1 else n

def rmsea_power(df, n, eps0=0.05, epsa=0.08, alpha=0.05):
    """epsa > eps0: test of close fit; epsa < eps0: test of not-close fit."""
    l0, la = _m(n) * df * eps0 ** 2, _m(n) * df * epsa ** 2
    if epsa > eps0:
        return 1 - ncx2.cdf(ncx2.ppf(1 - alpha, df, l0), df, la)
    return ncx2.cdf(ncx2.ppf(alpha, df, l0), df, la)

def min_n(df, target=0.80, **kw):
    g = lambda n: rmsea_power(df, n, **kw) - target
    return 5 if g(5) >= 0 else ceil(brentq(g, 5, 200000))

def reg_power(n, tested, total, f2=0.15, alpha=0.05):
    """Power of the joint test of `tested` predictors out of `total` predictors."""
    v = n - total - 1
    lam = f2 * (tested + v + 1) if REG_LAMBDA == "cohen" else f2 * n
    return 1 - ncf.cdf(f_dist.ppf(1 - alpha, tested, v), tested, v, lam)

def reg_min_n(tested, total, f2=0.15, alpha=0.05, power=0.90):
    g = lambda n: reg_power(n, tested, total, f2, alpha) - power
    return int(brentq(g, total + 3, 100000)) + 1

# ---------------- 2. FINAL MODEL ----------------
N = 405
FINAL = dict(
    items={"WR": 4, "RI": 5, "DIC": 4, "DIQ": 5, "SP": 4, "SA": 4,
           "RCC": 4, "SCR": 5, "EU": 5, "FSOC": 4, "PSVC": 4},          # 48 indicators
    exog=["WR", "DIC", "SP", "EU"],
    endo=["RI", "DIQ", "SA", "RCC", "SCR", "FSOC", "PSVC"],
    paths=[("WR", "RI"), ("DIC", "DIQ"), ("SP", "SA"),                    # H1, H3, H5
           ("RI", "RCC"), ("DIQ", "RCC"), ("SA", "RCC"),                  # H2, H4, H6
           ("RCC", "SCR"), ("EU", "SCR"),                                 # H7 (+ direct EU path)
           ("SCR", "FSOC"), ("FSOC", "PSVC"), ("SCR", "PSVC")],           # H15, H16, H17
    direct=[("WR", "RCC"), ("DIC", "RCC"), ("SP", "RCC")],                # sensitivity: direct paths
    # equation -> (tested predictors, total predictors); interactions: H12-H14 (RCC), H11 (SCR)
    equations={"RCC (3 interactions tested)": (3, 6), "SCR (EU x RCC tested)": (1, 3),
               "PSVC": (2, 2), "FSOC": (1, 1)},
)

# ---------------- 3. RUN ----------------
if __name__ == "__main__":
    m = FINAL
    print(f"N = {N} | N-1 in noncentrality: {USE_N_MINUS_1} | regression lambda: {REG_LAMBDA}\n")
    for label, extra in (("Hypothesised paths (no interactions)", []),
                         ("+ direct paths WR/DIC/SP -> RCC", m["direct"])):
        r = sem_df(m["items"], m["exog"], m["endo"], m["paths"] + extra)
        print(f"{label:<40} indicators={r['indicators']}  npar={r['npar']}  df={r['df']}  "
              f"CFA npar={r['cfa_npar']}  CFA df={r['cfa_df']}")
        print(f"{'':<40} power(close, eps0=.05 vs .08)={rmsea_power(r['df'], N):.3f}  "
              f"power(not-close)={rmsea_power(r['df'], N, 0.05, 0.01):.3f}  "
              f"minN(.80)={min_n(r['df'])}  N:npar={N / r['npar']:.2f}")
    r = sem_df(m["items"], m["exog"], m["endo"], m["paths"])
    print(f"\nCFA (measurement model): df={r['cfa_df']}  power(close)={rmsea_power(r['cfa_df'], N):.3f}  "
          f"minN(.80)={min_n(r['cfa_df'])}")
    print("\nRegression-based minimum N (f2 = .15, power = .90, alpha = .05):")
    for eq, (u, p) in m["equations"].items():
        print(f"  {eq:<30} tested={u} total={p}  min N = {reg_min_n(u, p)}   "
              f"power at N={N} for small effect (f2=.02) = {reg_power(N, u, p, 0.02):.3f}")

N = 405 | N-1 in noncentrality: True | regression lambda: cohen

Hypothesised paths (no interactions)     indicators=48  npar=113  df=1063  CFA npar=151  CFA df=1025
                                         power(close, eps0=.05 vs .08)=1.000  power(not-close)=1.000  minN(.80)=33  N:npar=3.58
+ direct paths WR/DIC/SP -> RCC          indicators=48  npar=116  df=1060  CFA npar=151  CFA df=1025
                                         power(close, eps0=.05 vs .08)=1.000  power(not-close)=1.000  minN(.80)=33  N:npar=3.49

CFA (measurement model): df=1025  power(close)=1.000  minN(.80)=33

Regression-based minimum N (f2 = .15, power = .90, alpha = .05):
  RCC (3 interactions tested)    tested=3 total=6  min N = 102   power at N=405 for small effect (f2=.02) = 0.652
  SCR (EU x RCC tested)          tested=1 total=3  min N = 75   power at N=405 for small effect (f2=.02) = 0.808
  PSVC                           tested=2 total=2  min N = 88   power at N=405 for small effect (f2=.02) = 0.720
  F